# ESG-Linked Executive Compensation Analysis

This notebook demonstrates how to use the ESG compensation classification pipeline with edgartools to analyze proxy statements.

In [ ]:
# Import required modules
import sys
sys.path.insert(0, '..')  # Add parent directory to path

from esg_compensation_pipeline_all_in_one import process_proxy_statement
from edgar import Company
import pandas as pd

## Example 1: Single Company Analysis

Let's analyze ESG compensation for a specific company.

In [ ]:
# Get company
company = Company("AAPL")
print(f"Analyzing: {company.name}")

In [ ]:
# Get recent proxy statements (DEF 14A)
def14a_filings = company.get_filings(form="DEF 14A").latest(3)
print(f"Found {len(def14a_filings)} DEF 14A filings")
for filing in def14a_filings:
    print(f"  - {filing.filing_date}: {filing.accession_number}")

In [ ]:
# Process the latest proxy statement
latest_filing = def14a_filings[0]
doc = latest_filing.primary_document

if doc:
    # Extract text (try both text and html methods depending on document type)
    try:
        proxy_text = doc.text()
    except:
        try:
            proxy_text = doc.html()
        except:
            proxy_text = str(doc)
    
    # Process with ESG classifier
    result = process_proxy_statement(
        firm_id=company.cik,
        year=latest_filing.filing_date.year,
        proxy_text=proxy_text
    )
    
    # Display results
    print("\n=== ESG Compensation Analysis ===")
    print(f"ESG-Linked Compensation: {'Yes' if result['ESG_PAY'] == 1 else 'No' if result['ESG_PAY'] == 0 else 'Unknown'}")
    print(f"Confidence: {result['ESG_PAY_confidence']}")
    print(f"Tier: {result['ESG_PAY_tier']}")
    print(f"\nMatched Terms: {', '.join(result['matched_terms'])}")
    print(f"\nESG Metrics Breakdown:")
    print(f"  - Environmental (E): {result['E_metrics']}")
    print(f"  - Social (S): {result['S_metrics']}")
    print(f"  - Governance (G): {result['G_metrics']}")
    print(f"\nESG Intensity: {result['ESG_PAY_INTENSITY']}")
    print(f"ESG Breadth: {result['ESG_PAY_BREADTH']} categories")
    print(f"Weighted Score: {result['ESG_PAY_SCORE']:.2f}")
    print(f"\nApplies to CEO: {result['applies_to_ceo']}")
    print(f"Mentions NEOs: {result['mentions_neo']}")
else:
    print("Could not extract document text")

## Example 2: Multi-Year Trend Analysis

Analyze ESG compensation trends over multiple years.

In [ ]:
# Process multiple years
results = []

for filing in def14a_filings:
    doc = filing.primary_document
    if doc:
        try:
            proxy_text = doc.text()
        except:
            try:
                proxy_text = doc.html()
            except:
                continue
        
        result = process_proxy_statement(
            firm_id=company.cik,
            year=filing.filing_date.year,
            proxy_text=proxy_text
        )
        results.append(result)

# Create DataFrame
df = pd.DataFrame(results)
df_display = df[['year', 'ESG_PAY', 'ESG_PAY_confidence', 'ESG_PAY_INTENSITY', 
                  'E_metrics', 'S_metrics', 'G_metrics', 'ESG_PAY_SCORE']]
print(df_display.to_string(index=False))

## Example 3: Multi-Company Comparison

Compare ESG compensation practices across companies.

In [ ]:
# Analyze multiple companies
companies = ["AAPL", "MSFT", "GOOGL", "TSLA"]
comparison_results = []

for ticker in companies:
    try:
        comp = Company(ticker)
        filings = comp.get_filings(form="DEF 14A").latest(1)
        
        if len(filings) > 0:
            filing = filings[0]
            doc = filing.primary_document
            
            if doc:
                try:
                    proxy_text = doc.text()
                except:
                    try:
                        proxy_text = doc.html()
                    except:
                        continue
                
                result = process_proxy_statement(
                    firm_id=comp.cik,
                    year=filing.filing_date.year,
                    proxy_text=proxy_text
                )
                result['ticker'] = ticker
                result['company_name'] = comp.name
                comparison_results.append(result)
    except Exception as e:
        print(f"Error processing {ticker}: {e}")

# Display comparison
if comparison_results:
    comp_df = pd.DataFrame(comparison_results)
    comp_display = comp_df[['ticker', 'company_name', 'year', 'ESG_PAY', 
                              'ESG_PAY_confidence', 'ESG_PAY_INTENSITY', 'ESG_PAY_SCORE']]
    print(comp_display.to_string(index=False))

## Example 4: Demo with Synthetic Data

Test the classifier with example proxy statement text.

In [ ]:
# Example 1: Strong ESG linkage
demo_text_1 = """
Compensation Discussion and Analysis

Our executive compensation program includes ESG-linked pay components. 
The annual bonus plan incorporates an ESG modifier based on our performance 
against sustainability metrics.

For the CEO and other named executive officers, we have established the following 
ESG metrics:
- Emissions reduction targets (30% weight)
- Diversity goals for leadership positions (20% weight)  
- Safety performance measured by TRIR (20% weight)
- Employee engagement scores (15% weight)

The long-term incentive plan (LTIP) includes climate targets and renewable 
energy investment goals.
"""

result_1 = process_proxy_statement("DEMO_1", 2024, demo_text_1)
print("Example 1: Strong ESG Linkage")
print(f"  ESG_PAY: {result_1['ESG_PAY']}")
print(f"  Confidence: {result_1['ESG_PAY_confidence']}")
print(f"  Matched terms: {result_1['matched_terms'][:5]}")
print(f"  Score: {result_1['ESG_PAY_SCORE']:.2f}\n")

# Example 2: No ESG linkage
demo_text_2 = """
Compensation Discussion and Analysis

Our executive compensation is based on financial performance metrics. The CEO and 
named executive officers receive compensation tied to:
- Revenue growth
- Earnings per share (EPS)
- Return on equity (ROE)
- Total shareholder return (TSR)

Annual incentive bonuses are calculated based on achievement against these 
financial targets.
"""

result_2 = process_proxy_statement("DEMO_2", 2024, demo_text_2)
print("Example 2: No ESG Linkage")
print(f"  ESG_PAY: {result_2['ESG_PAY']}")
print(f"  Confidence: {result_2['ESG_PAY_confidence']}")
print(f"  Score: {result_2['ESG_PAY_SCORE']:.2f}\n")

# Example 3: Greenwashing (generic ESG mentions without compensation linkage)
demo_text_3 = """
Compensation Discussion and Analysis

We are committed to sustainability and environmental stewardship. Our board has 
established an ESG committee to oversee sustainability initiatives.

Executive compensation is based on financial performance. Annual bonuses are tied 
to revenue and profit targets.
"""

result_3 = process_proxy_statement("DEMO_3", 2024, demo_text_3)
print("Example 3: Greenwashing Detection")
print(f"  ESG_PAY: {result_3['ESG_PAY']}")
print(f"  Confidence: {result_3['ESG_PAY_confidence']}")
print(f"  Governance flag: {result_3['governance_vs_compensation_flag']}")
print(f"  Score: {result_3['ESG_PAY_SCORE']:.2f}")

## Example 5: Batch Processing

Process multiple companies efficiently.

In [ ]:
def analyze_company_esg_compensation(ticker: str, years: int = 3):
    """Analyze ESG compensation for a company over multiple years."""
    try:
        company = Company(ticker)
        filings = company.get_filings(form="DEF 14A").latest(years)
        results = []
        
        for filing in filings:
            doc = filing.primary_document
            if not doc:
                continue
                
            try:
                proxy_text = doc.text()
            except:
                try:
                    proxy_text = doc.html()
                except:
                    continue
            
            result = process_proxy_statement(
                firm_id=company.cik,
                year=filing.filing_date.year,
                proxy_text=proxy_text
            )
            result['ticker'] = ticker
            results.append(result)
        
        return results
    except Exception as e:
        print(f"Error analyzing {ticker}: {e}")
        return []

# Analyze multiple companies
tech_companies = ["AAPL", "MSFT", "GOOGL"]
all_results = []

for ticker in tech_companies:
    results = analyze_company_esg_compensation(ticker, years=2)
    all_results.extend(results)

# Create summary
if all_results:
    summary_df = pd.DataFrame(all_results)
    print("\n=== ESG Compensation Summary ===")
    print(summary_df[['ticker', 'year', 'ESG_PAY', 'ESG_PAY_confidence', 
                       'ESG_PAY_INTENSITY']].to_string(index=False))